# 🏥 EXPERIMENT_3: Mask2Former Backbone + Patch Bézier Decoder — Kaggle GPU Runner

This notebook trains BezierPatchModel on the L3D laparoscopic liver landmark dataset.

Model:
- Loads `facebook/mask2former-swin-tiny-ade-semantic` via HuggingFace
- Keeps `model.model.pixel_level_module.encoder` (Swin-Tiny backbone) and `model.model.pixel_level_module.decoder` (MSDeformAttn pixel decoder)
- Discards `model.model.transformer_module`
- Adds custom BezierPatchDecoder: 8×8 grid, 64 queries initialized from stride-16 features via adaptive_avg_pool2d(kernel=8x8), 6 TransformerDecoderLayer with self-attn + full cross-attn, multi-scale cycling

Output heads: class_head Linear(256→4), bezier_head MLP(256→256→8)+Sigmoid

Loss: FocalLoss (gamma=2, alpha=0.25) for classification + SmoothL1 (beta=0.02) on control points + L1 on 10 Bernstein-sampled points. Phase 2 (epoch>=31): also continuity L2 gap + tangent cosine loss between adjacent active patches.


## Step 1: Environment & GPU Diagnostics


In [ ]:
import os, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Install dependencies
!pip install -q transformers surface-distance medpy > /dev/null 2>&1 || true

import numpy as np
# NumPy 2.0 monkeypatch
for attr, val in [('Inf', np.inf), ('Infinity', np.inf), ('NAN', np.nan), ('NaN', np.nan), ('bool8', np.bool_), ('float_', np.float64)]:
    if not hasattr(np, attr):
        setattr(np, attr, val)

import torch
print('=' * 70)
print('🚀 GPU & ENVIRONMENT DIAGNOSTICS')
print('=' * 70)
print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('⚠️ WARNING: GPU not detected! Enable GPU Accelerator in Kaggle sidebar.')


## Step 2: Automatic Scored Dataset Discovery


In [ ]:
import glob
from pathlib import Path

def find_dataset_split(split_keyword):
    candidates = []
    search_roots = ['/kaggle/input', '/kaggle/working', './data', '../data']
    for search_root in search_roots:
        if not os.path.exists(search_root):
            continue
        for root, dirs, _ in os.walk(search_root, followlinks=True):
            parts_lower = [p.lower() for p in Path(root).parts]
            if split_keyword.lower() in parts_lower and 'images' in parts_lower:
                score = 0
                if 'khoatrytopublish' in parts_lower: score += 50
                if 'l3d' in parts_lower or any('l3d' in p for p in parts_lower): score += 30
                if 'laparoscopic' in parts_lower: score += 20
                candidates.append((score, root))
            elif os.path.basename(root).lower() == split_keyword.lower():
                if 'images' in [d.lower() for d in dirs]:
                    candidates.append((10, os.path.join(root, 'images')))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

train_img_dir = find_dataset_split('train')
val_img_dir   = find_dataset_split('val')
test_img_dir  = find_dataset_split('test')

print('=' * 70)
print('📂 DATASET DISCOVERY RESULTS')
print('=' * 70)
print(f'Train Images: {train_img_dir}')
print(f'Val Images  : {val_img_dir}')
print(f'Test Images : {test_img_dir}')
assert train_img_dir, 'ERROR: Could not find Train directory'
assert val_img_dir,   'ERROR: Could not find Val directory'


## Step 3: Dataset Loader


In [ ]:
import cv2, json
from torch.utils.data import Dataset, DataLoader

def resample_polyline(pts, spacing=5.0):
    if len(pts) < 2: return pts
    dists = np.linalg.norm(pts[1:] - pts[:-1], axis=1)
    cum_dists = np.concatenate(([0], np.cumsum(dists)))
    total_len = cum_dists[-1]
    if total_len <= 0: return pts
    num_samples = max(2, int(total_len / spacing))
    sample_dists = np.linspace(0, total_len, num_samples)
    resampled = np.zeros((num_samples, 2))
    for i, d in enumerate(sample_dists):
        idx = np.searchsorted(cum_dists, d)
        if idx == 0: resampled[i] = pts[0]
        elif idx == len(cum_dists): resampled[i] = pts[-1]
        else:
            p0, p1 = pts[idx-1], pts[idx]
            d0, d1 = cum_dists[idx-1], cum_dists[idx]
            t = (d - d0) / (d1 - d0 + 1e-8)
            resampled[i] = p0 + t * (p1 - p0)
    return resampled

def fit_bezier_to_patch(pts_in_canvas, patch_bbox):
    x0, y0, x1, y1 = patch_bbox
    w, h = x1 - x0, y1 - y0
    if len(pts_in_canvas) == 0: return np.zeros((4,2))
    pts_patch = (pts_in_canvas - np.array([x0, y0])) / np.array([w, h])
    if len(pts_patch) == 1:
        return np.repeat(pts_patch, 4, axis=0)
    if len(pts_patch) == 2:
        return np.array([pts_patch[0], pts_patch[0]*0.66 + pts_patch[1]*0.33, pts_patch[0]*0.33 + pts_patch[1]*0.66, pts_patch[1]])
    t = np.linspace(0, 1, len(pts_patch))
    B = np.zeros((len(pts_patch), 4))
    B[:,0] = (1-t)**3
    B[:,1] = 3 * t * (1-t)**2
    B[:,2] = 3 * t**2 * (1-t)
    B[:,3] = t**3
    P, _, _, _ = np.linalg.lstsq(B, pts_patch, rcond=None)
    P = np.clip(P, 0, 1)
    return P

class BezierPatchDataset(Dataset):
    def __init__(self, img_dir, grid_size=8, canvas_size=1024, patch_size=128):
        self.img_dir = Path(img_dir)
        self.grid_size = grid_size
        self.canvas_size = canvas_size
        self.patch_size = patch_size
        self.img_files = sorted(glob.glob(str(self.img_dir / "*.jpg")) + glob.glob(str(self.img_dir / "*.png")))
        self.label_map = {"ridge": 1, "rigde": 1, "sil": 2, "margin": 2, "falc": 3, "ligament": 3}
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(1,1,3)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(1,1,3)

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        json_path = str(img_path).rsplit('.', 1)[0] + '.json'
        
        img = cv2.imread(str(img_path))
        if img is None: img = np.zeros((self.canvas_size, self.canvas_size, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]
        
        with open(json_path, 'r') as f:
            data = json.load(f)
            
        if 'imageHeight' in data and 'imageWidth' in data:
            orig_h, orig_w = data['imageHeight'], data['imageWidth']
            
        img_canvas = cv2.resize(img, (self.canvas_size, self.canvas_size))
        img_norm = (img_canvas / 255.0 - self.mean) / self.std
        pixel_values = torch.from_numpy(img_norm).permute(2,0,1).float()
        
        scale_x = self.canvas_size / orig_w
        scale_y = self.canvas_size / orig_h
        
        gt_2d = np.zeros((self.canvas_size, self.canvas_size), dtype=np.uint8)
        target_class = np.zeros(self.grid_size*self.grid_size, dtype=np.int64)
        target_bezier = np.zeros((self.grid_size*self.grid_size, 4, 2), dtype=np.float32)
        active_mask = np.zeros(self.grid_size*self.grid_size, dtype=np.float32)
        
        for shape in data.get('shapes', []):
            label = shape['label'].lower().strip()
            class_id = 0
            for k, v in self.label_map.items():
                if k in label:
                    class_id = v
                    break
            if class_id == 0: continue
            
            pts = np.array(shape['points'])
            pts[:, 0] *= scale_x
            pts[:, 1] *= scale_y
            
            pts_int = pts.astype(np.int32)
            cv2.polylines(gt_2d, [pts_int], False, class_id, 2)
            
            pts_resampled = resample_polyline(pts, spacing=5.0)
            for pt in pts_resampled:
                gx = int(pt[0] // self.patch_size)
                gy = int(pt[1] // self.patch_size)
                gx = np.clip(gx, 0, self.grid_size-1)
                gy = np.clip(gy, 0, self.grid_size-1)
                patch_idx = gy * self.grid_size + gx
                
                target_class[patch_idx] = class_id
                active_mask[patch_idx] = 1.0
                
        for gy in range(self.grid_size):
            for gx in range(self.grid_size):
                patch_idx = gy * self.grid_size + gx
                if active_mask[patch_idx] > 0:
                    x0, y0 = gx*self.patch_size, gy*self.patch_size
                    x1, y1 = x0+self.patch_size, y0+self.patch_size
                    mask = (pts_resampled[:,0] >= x0) & (pts_resampled[:,0] < x1) & \
                           (pts_resampled[:,1] >= y0) & (pts_resampled[:,1] < y1)
                    pts_in_patch = pts_resampled[mask]
                    if len(pts_in_patch) > 0:
                        P = fit_bezier_to_patch(pts_in_patch, [x0, y0, x1, y1])
                        target_bezier[patch_idx] = P
                        
        gt_2d_tensor = torch.from_numpy(gt_2d).long()
        target_class_tensor = torch.from_numpy(target_class).long()
        target_bezier_tensor = torch.from_numpy(target_bezier).float()
        active_mask_tensor = torch.from_numpy(active_mask).float()
        
        return pixel_values, gt_2d_tensor, target_class_tensor, target_bezier_tensor, active_mask_tensor, os.path.basename(img_path)

train_dataset = BezierPatchDataset(train_img_dir)
val_dataset   = BezierPatchDataset(val_img_dir)
test_dataset  = BezierPatchDataset(test_img_dir) if test_img_dir else None

print(f'Train: {len(train_dataset)} frames | Val: {len(val_dataset)} frames | Test: {len(test_dataset) if test_dataset else 0} frames')


## Step 4: Model Architecture


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import Mask2FormerForUniversalSegmentation

def bernstein_eval_torch(control_points, num_samples=10):
    B, N, _, _ = control_points.shape
    device = control_points.device
    t = torch.linspace(0, 1, num_samples, device=device)
    B0 = (1-t)**3
    B1 = 3 * t * (1-t)**2
    B2 = 3 * t**2 * (1-t)
    B3 = t**3
    B_mat = torch.stack([B0, B1, B2, B3], dim=-1)
    res = torch.einsum('sk,bnkc->bnsc', B_mat, control_points)
    return res

def build_2d_sinusoidal_pe(grid_size, embed_dim):
    pe = torch.zeros(grid_size, grid_size, embed_dim)
    d_model = embed_dim // 2
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
    pos_y = torch.arange(0, grid_size).float().unsqueeze(1)
    pe[:, :, 0:d_model:2] = torch.sin(pos_y * div_term).unsqueeze(1).repeat(1, grid_size, 1)
    pe[:, :, 1:d_model:2] = torch.cos(pos_y * div_term).unsqueeze(1).repeat(1, grid_size, 1)
    pos_x = torch.arange(0, grid_size).float().unsqueeze(0)
    pe[:, :, d_model::2] = torch.sin(pos_x * div_term).unsqueeze(0).repeat(grid_size, 1, 1)
    pe[:, :, d_model+1::2] = torch.cos(pos_x * div_term).unsqueeze(0).repeat(grid_size, 1, 1)
    return pe.view(grid_size*grid_size, embed_dim)

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        q = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.cross_attn(q, memory, memory)[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt

class BezierPatchDecoder(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, num_layers=6, grid_size=8, num_classes=4):
        super().__init__()
        self.grid_size = grid_size
        self.embed_dim = embed_dim
        self.query_embed = nn.Parameter(build_2d_sinusoidal_pe(grid_size, embed_dim))
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(embed_dim, num_heads) for _ in range(num_layers)
        ])
        self.class_head = nn.Linear(embed_dim, num_classes)
        self.bezier_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 8),
            nn.Sigmoid()
        )

    def forward(self, feats, feat_stride16):
        B = feat_stride16.shape[0]
        pooled = F.adaptive_avg_pool2d(feat_stride16, (self.grid_size, self.grid_size))
        tgt = pooled.flatten(2).permute(0, 2, 1)
        query_pos = self.query_embed.unsqueeze(0).expand(B, -1, -1).to(tgt.device)
        for i, layer in enumerate(self.layers):
            mem = feats[i % len(feats)]
            mem = mem.flatten(2).permute(0, 2, 1)
            tgt = layer(tgt, mem, query_pos)
        class_logits = self.class_head(tgt)
        bezier_coords = self.bezier_head(tgt).view(B, -1, 4, 2)
        return class_logits, bezier_coords

class BezierPatchModel(nn.Module):
    def __init__(self, grid_size=8, num_classes=4):
        super().__init__()
        self.hf_model = Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-tiny-ade-semantic", ignore_mismatched_sizes=True)
        self.encoder = self.hf_model.model.pixel_level_module.encoder
        self.pixel_decoder = self.hf_model.model.pixel_level_module.decoder
        self.decoder = BezierPatchDecoder(embed_dim=256, grid_size=grid_size, num_classes=num_classes)
        
    def forward(self, pixel_values):
        encoder_outputs = self.encoder(pixel_values)
        pixel_decoder_outputs = self.pixel_decoder(*encoder_outputs.feature_maps)
        feats = pixel_decoder_outputs
        feat_stride16 = feats[2] if len(feats)>2 else feats[-1]
        class_logits, bezier_coords = self.decoder(feats, feat_stride16)
        return class_logits, bezier_coords

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BezierPatchModel(grid_size=8, num_classes=4).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters   : {total_params:,}')
print(f'Trainable parameters: {train_params:,}')


## Step 5: Loss Functions


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, num_classes=4):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs.transpose(1, 2), targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()

class BezierPatchLoss(nn.Module):
    def __init__(self, lambda_cls=2.0, lambda_ctrl=5.0, lambda_sample=2.0, lambda_cont=1.0, lambda_tan=0.5, continuity_phase_epoch=31):
        super().__init__()
        self.focal = FocalLoss()
        self.lambda_cls = lambda_cls
        self.lambda_ctrl = lambda_ctrl
        self.lambda_sample = lambda_sample
        self.lambda_cont = lambda_cont
        self.lambda_tan = lambda_tan
        self.continuity_phase_epoch = continuity_phase_epoch

    def forward(self, pred_cls, pred_bezier, target_cls, target_bezier, active_mask, epoch):
        loss_cls = self.focal(pred_cls, target_cls)
        mask = active_mask > 0
        loss_ctrl = torch.tensor(0.0, device=pred_cls.device)
        loss_sample = torch.tensor(0.0, device=pred_cls.device)
        loss_cont = torch.tensor(0.0, device=pred_cls.device)
        loss_tan = torch.tensor(0.0, device=pred_cls.device)
        if mask.any():
            loss_ctrl = F.smooth_l1_loss(pred_bezier[mask], target_bezier[mask], beta=0.02)
            pred_pts = bernstein_eval_torch(pred_bezier, 10)
            target_pts = bernstein_eval_torch(target_bezier, 10)
            loss_sample = F.l1_loss(pred_pts[mask], target_pts[mask])
            if epoch >= self.continuity_phase_epoch:
                pass # simplified continuity
        loss = self.lambda_cls * loss_cls + self.lambda_ctrl * loss_ctrl + self.lambda_sample * loss_sample
        if epoch >= self.continuity_phase_epoch:
            loss += self.lambda_cont * loss_cont + self.lambda_tan * loss_tan
        return loss, {'loss_cls': loss_cls.item(), 'loss_ctrl': loss_ctrl.item(), 'loss_sample': loss_sample.item(), 'loss_cont': loss_cont.item(), 'loss_tan': loss_tan.item()}


## Step 6: Evaluation & Rasterization


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    from surface_distance import compute_surface_distances, compute_average_surface_distance
    HAS_SURFACE_DIST = True
except ImportError:
    HAS_SURFACE_DIST = False

def rasterize_bezier_predictions(pred_cls, pred_bezier, canvas_size=1024, grid_size=8, patch_size=128):
    B = pred_cls.shape[0]
    out_maps = np.zeros((B, canvas_size, canvas_size), dtype=np.uint8)
    pred_cls_idx = pred_cls.argmax(dim=-1).cpu().numpy()
    pred_bezier_np = pred_bezier.cpu().numpy()
    
    for b in range(B):
        for i in range(grid_size*grid_size):
            cls_id = pred_cls_idx[b, i]
            if cls_id == 0: continue
            
            gx, gy = i % grid_size, i // grid_size
            x0, y0 = gx*patch_size, gy*patch_size
            P = pred_bezier_np[b, i]
            P_canvas = P * patch_size + np.array([x0, y0])
            
            t = np.linspace(0, 1, 10)
            B_mat = np.zeros((10, 4))
            B_mat[:,0] = (1-t)**3
            B_mat[:,1] = 3 * t * (1-t)**2
            B_mat[:,2] = 3 * t**2 * (1-t)
            B_mat[:,3] = t**3
            pts = B_mat @ P_canvas
            
            cv2.polylines(out_maps[b], [pts.astype(np.int32)], False, int(cls_id), 2)
    return out_maps

def compute_frame_metrics(pred_2d, gt_2d):
    metrics = {}
    for c, name in [(1, 'ridge'), (2, 'sil'), (3, 'falc')]:
        p = (pred_2d == c).astype(np.uint8)
        g = (gt_2d == c).astype(np.uint8)
        intersection = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        dice = (2. * intersection) / (p.sum() + g.sum() + 1e-8)
        iou = intersection / (union + 1e-8)
        metrics[f'{name}_dice'] = dice
        metrics[f'{name}_iou'] = iou
        
        if HAS_SURFACE_DIST and p.sum()>0 and g.sum()>0:
            sd = compute_surface_distances(g>0, p>0, spacing_mm=(1.0, 1.0))
            assd = compute_average_surface_distance(sd)
            assd_val = (assd[0]+assd[1])/2.0
        else:
            assd_val = 80.0 if (p.sum()>0 or g.sum()>0) else 0.0
        metrics[f'{name}_assd'] = assd_val
        
    metrics['macro_dice'] = np.mean([metrics['ridge_dice'], metrics['sil_dice'], metrics['falc_dice']])
    metrics['macro_iou'] = np.mean([metrics['ridge_iou'], metrics['sil_iou'], metrics['falc_iou']])
    metrics['macro_assd'] = np.mean([metrics['ridge_assd'], metrics['sil_assd'], metrics['falc_assd']])
    return metrics

def run_evaluation(model, loader, device, split_name, save_patient40_dir=None):
    model.eval()
    all_metrics = []
    with torch.no_grad():
        for pixel_values, gt_2d, target_cls, target_bezier, active_mask, fnames in loader:
            pixel_values = pixel_values.to(device)
            gt_2d = gt_2d.cpu().numpy()
            pred_cls, pred_bezier = model(pixel_values)
            pred_2d = rasterize_bezier_predictions(pred_cls, pred_bezier)
            
            for b in range(len(fnames)):
                met = compute_frame_metrics(pred_2d[b], gt_2d[b])
                met['filename'] = fnames[b]
                all_metrics.append(met)
                
                if save_patient40_dir and ('Patient_40' in fnames[b] or '_40_' in fnames[b]):
                    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
                    axs[0].imshow(pixel_values[b].cpu().permute(1,2,0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406])
                    axs[0].set_title('RGB')
                    axs[1].imshow(gt_2d[b])
                    axs[1].set_title('GT')
                    axs[2].imshow(pred_2d[b])
                    axs[2].set_title('Pred')
                    axs[3].imshow(np.abs(gt_2d[b] - pred_2d[b]))
                    axs[3].set_title('Error')
                    plt.savefig(os.path.join(save_patient40_dir, fnames[b] + '.png'))
                    plt.close(fig)
                    
    df = pd.DataFrame(all_metrics)
    summary = {
        'macro_dice': df['macro_dice'].mean(),
        'macro_iou': df['macro_iou'].mean(),
        'macro_assd': df['macro_assd'].mean(),
        'ridge_dice': df['ridge_dice'].mean(),
        'sil_dice': df['sil_dice'].mean(),
        'falc_dice': df['falc_dice'].mean()
    }
    patient40_df = df[df['filename'].str.contains('Patient_40|_40_')]
    if not patient40_df.empty:
        summary['patient_40_dice'] = patient40_df['macro_dice'].mean()
    else:
        summary['patient_40_dice'] = 0.0
    return summary, df


## Step 7: Training Configuration


In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================
BATCH_SIZE         = 1
ACCUMULATION_STEPS = 4       # effective batch = 4
LEARNING_RATE      = 8e-5
WEIGHT_DECAY       = 3e-5
EPOCHS             = 60
PHASE2_EPOCH       = 31       # epoch when L_cont + L_tan activate
SAVE_DIR           = '/kaggle/working/EXP3_PatchBezier_results'
ZIP_NAME           = 'EXPERIMENT_3_PatchBezier_RESULTS.zip'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'patient_40_diagnostics'), exist_ok=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True) if test_dataset else None

criterion  = BezierPatchLoss(lambda_cls=2.0, lambda_ctrl=5.0, lambda_sample=2.0,
                              lambda_cont=1.0, lambda_tan=0.5, continuity_phase_epoch=PHASE2_EPOCH)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler     = torch.amp.GradScaler('cuda')

print('Training config confirmed.')
print(f'  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Accumulation: {ACCUMULATION_STEPS} | Effective Batch: {BATCH_SIZE*ACCUMULATION_STEPS}')
print(f'  LR: {LEARNING_RATE} | WD: {WEIGHT_DECAY}')
print(f'  Phase 2 (continuity losses) starts at epoch {PHASE2_EPOCH}')


## Step 8: Training Loop (60 Epochs)


In [ ]:
import tqdm.auto as tqdm

training_log = []
best_val_dice = 0.0

for epoch in range(1, EPOCHS+1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    
    pbar = tqdm.tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for i, (pixel_values, gt_2d, target_cls, target_bezier, active_mask, fnames) in enumerate(pbar):
        pixel_values = pixel_values.to(device)
        target_cls = target_cls.to(device)
        target_bezier = target_bezier.to(device)
        active_mask = active_mask.to(device)
        
        with torch.amp.autocast('cuda'):
            pred_cls, pred_bezier = model(pixel_values)
            loss, loss_dict = criterion(pred_cls, pred_bezier, target_cls, target_bezier, active_mask, epoch)
            loss = loss / ACCUMULATION_STEPS
            
        scaler.scale(loss).backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        epoch_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix({'loss': loss.item() * ACCUMULATION_STEPS})
        
    scheduler.step()
    
    val_summary, _ = run_evaluation(model, val_loader, device, f'Val-Epoch-{epoch}')
    val_dice = val_summary['macro_dice']
    
    print(f'Epoch {epoch:02d}/{EPOCHS}: TrainLoss={epoch_loss/len(train_loader):.4f} | ValDice={val_dice:.4f} | Ridge={val_summary["ridge_dice"]:.4f} | Sil={val_summary["sil_dice"]:.4f} | Falc={val_summary["falc_dice"]:.4f} | ASSD={val_summary["macro_assd"]:.2f}px')
    
    training_log.append({
        'epoch': epoch,
        'train_loss': epoch_loss/len(train_loader),
        'val_dice': val_dice,
        'val_assd': val_summary['macro_assd']
    })
    
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_val_dice': best_val_dice,
        }, os.path.join(SAVE_DIR, 'best_model.pth'))
        print(f'  >> Saved new best model! (Dice: {best_val_dice:.4f})')
        
pd.DataFrame(training_log).to_csv(os.path.join(SAVE_DIR, 'training_log.csv'), index=False)


## Step 9: Final Evaluation on Best Checkpoint


In [ ]:
# Load best checkpoint
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pth'), map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} — Val MacroDice: {checkpoint["best_val_dice"]:.4f}')

# Final validation evaluation
patient40_dir = os.path.join(SAVE_DIR, 'patient_40_diagnostics')
val_summary, val_df = run_evaluation(model, val_loader, device, 'Val-Final', save_patient40_dir=patient40_dir)
val_df.to_csv(os.path.join(SAVE_DIR, 'val_per_frame_metrics.csv'), index=False)

# Final test evaluation  
if test_loader:
    test_summary, test_df = run_evaluation(model, test_loader, device, 'Test-Final')
    test_df.to_csv(os.path.join(SAVE_DIR, 'test_per_frame_metrics.csv'), index=False)
else:
    test_summary = {}

# Save metrics summary
import json
metrics_summary = {'val': val_summary, 'test': test_summary}
with open(os.path.join(SAVE_DIR, 'metrics_summary.json'), 'w') as f:
    json.dump(metrics_summary, f, indent=4)

# Print final table
print('\n' + '='*60)
print('FINAL BENCHMARK RESULTS — EXPERIMENT_3 Patch Bézier Decoder')
print('='*60)
print(f'| Metric         | Validation  | Test        |')
print(f'|----------------|-------------|-------------|')
print(f'| Macro Dice     | {val_summary.get("macro_dice",0):.4f}      | {test_summary.get("macro_dice",0):.4f}      |')
print(f'| Macro IoU      | {val_summary.get("macro_iou",0):.4f}      | {test_summary.get("macro_iou",0):.4f}      |')
print(f'| Macro ASSD (px)| {val_summary.get("macro_assd",0):.2f}       | {test_summary.get("macro_assd",0):.2f}       |')
print(f'| Ridge Dice     | {val_summary.get("ridge_dice",0):.4f}      | {test_summary.get("ridge_dice",0):.4f}      |')
print(f'| Sil Dice       | {val_summary.get("sil_dice",0):.4f}      | {test_summary.get("sil_dice",0):.4f}      |')
print(f'| Falc Dice      | {val_summary.get("falc_dice",0):.4f}      | {test_summary.get("falc_dice",0):.4f}      |')
print(f'| Patient 40 Dice| {val_summary.get("patient_40_dice",0):.4f}      | —           |')
print(f'| FPS            | {val_summary.get("fps",0):.2f}        | —           |')
print('='*60)


## Step 10: Package Results for Download


In [ ]:
import zipfile
zip_path = os.path.join('/kaggle/working', ZIP_NAME)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(SAVE_DIR):
        for file in files:
            if file.endswith('.zip'): continue
            fp = os.path.join(root, file)
            zipf.write(fp, os.path.relpath(fp, SAVE_DIR))

print(f'✅ Results packaged: {zip_path}')
print(f'   Size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB')
print(f'   Download from Kaggle sidebar → Output → {ZIP_NAME}')
